# Решения: практика transform

**Для преподавателя.** Все задачи из `lesson.ipynb` и `homework.ipynb`.

In [ ]:
LAB_SCORES = [72, 81, 55, 90, 44, 38, 92, 58]
AREAS_SQM = [28, 32, 45, 55, 60]
ROOMS_COUNTS = [1, 1, 2, 2, 3]
FEATURE_ROWS = [
    (28, 1, 3, 'north'),
    (45, 2, 7, 'center'),
    (60, 3, 12, 'south'),
    (32, 1, 2, 'west'),
]


def describe_numbers(values):
    if not values:
        return (0.0, 0.0, 0.0, 0)
    return (sum(values) / len(values), min(values), max(values), len(values))


def min_max_scale(values):
    if not values:
        return []
    lo, hi = min(values), max(values)
    if lo == hi:
        return [0.0] * len(values)
    return [(x - lo) / (hi - lo) for x in values]


def clip_outlier(x, low, high):
    return max(low, min(high, x))


def grade_stats(scores, pass_threshold=60):
    passed = sum(1 for s in scores if s >= pass_threshold)
    failed = len(scores) - passed
    mean = sum(scores) / len(scores) if scores else 0.0
    return (mean, passed, failed)


## Урок. 2. Describe `LAB_SCORES`

In [ ]:
stats = describe_numbers(LAB_SCORES)
assert stats[3] == 8
print(stats)

## Урок. 3. Два scale

In [ ]:
scaled_areas = min_max_scale(AREAS_SQM)
scaled_rooms = min_max_scale(ROOMS_COUNTS)
assert abs(max(scaled_areas) - 1) < 1e-9
assert abs(max(scaled_rooms) - 1) < 1e-9

## Урок. 4. Порог для k=4

In [ ]:
TARGET_PASSED = 4
best_threshold = None
for threshold in range(0, 101):
    _, passed, _ = grade_stats(LAB_SCORES, threshold)
    if passed == TARGET_PASSED:
        best_threshold = threshold
        break
assert best_threshold == 59
print('порог =', best_threshold)

## Урок. 5. `apply_transform`

In [ ]:
def apply_transform(values, fn):
    return fn(values)


after_stats = describe_numbers(apply_transform(LAB_SCORES, min_max_scale))
assert abs(after_stats[1] - 0) < 1e-9 and abs(after_stats[2] - 1) < 1e-9

## ДЗ. 2. `numeric_stats_from_row`

In [ ]:
def numeric_stats_from_row(row):
    nums = [x for x in row if isinstance(x, (int, float))]
    return describe_numbers(nums)[0], max(nums)


mean, mx = numeric_stats_from_row(FEATURE_ROWS[0])
assert abs(mean - 10.666666) < 1e-5 and mx == 28

## ДЗ. 3. `transform_pipeline`

In [ ]:
def clip_scores(scores, low, high):
    return [clip_outlier(s, low, high) for s in scores]


def transform_pipeline(values, *fns):
    result = values
    for fn in fns:
        result = fn(result)
    return result


only_scale = describe_numbers(min_max_scale(LAB_SCORES))[0]
clip_then_scale = describe_numbers(
    transform_pipeline(LAB_SCORES, lambda xs: clip_scores(xs, 50, 95), min_max_scale)
)[0]
print('mean: только scale', only_scale, '; clip→scale', clip_then_scale)

## ДЗ. 4. Сравнение preprocess

In [ ]:
before = describe_numbers(LAB_SCORES)[0]
after_clip = describe_numbers(clip_scores(LAB_SCORES, 50, 95))[0]
print('mean до:', before, 'после clip [50,95]:', after_clip)